<a href="https://colab.research.google.com/github/mschemerii/cosc-650-applied-llm-systems/blob/week4-assignment-agent-tool-loop/week-04/week4_tool_calling_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 4 — Tool Calling: From Model to Agent

**COSC 650 — Applied LLM Systems**

This notebook implements an end-to-end function-calling loop using the Gemini Flash model through Google's OpenAI-compatible endpoint. The assistant can choose among three constrained tools, receive tool results, and continue reasoning until it produces a final answer.

## Assignment coverage

- **Part 1:** Three tools with constrained JSON schemas using enums, required fields, and explicit types.
- **Part 2:** A guarded code-runner that permits arithmetic expressions only, validates the model-supplied code before execution, and enforces a time limit.
- **Part 3:** Evaluation queries covering all three tools, including a two-tool sequence, with a visible tool-call log.
- **Part 4:** A real runtime failure (`10 / 0`) that returns a structured error and is recovered through a retry.
- **Part 5:** Repository documentation and GitHub issue #15 document the design and failure/recovery case.

The live Gemini calls and reported outputs should be generated by running this notebook. Do not treat unexecuted example text as measured results.

## 1. Setup

The assignment specifies Gemini Flash through the OpenAI-compatible endpoint. The API key is loaded without being written into the notebook.

In Google Colab, store `GEMINI_API_KEY` in the **Secrets** panel. The setup cell first checks the environment and then falls back to Colab Secrets. The same notebook can therefore run in Colab or a local Jupyter environment.

The default model is `gemini-3.5-flash-lite`, a stable Flash-Lite model suited to high-volume function-calling work. You can override it without editing the notebook by setting a `GEMINI_MODEL` environment variable.

In [1]:
!pip -q install openai

In [ ]:
import ast
import json
import math
import multiprocessing as mp
import os
import time
from typing import Any

from openai import OpenAI

# Read the API key from a normal environment variable first.
API_KEY = os.environ.get("GEMINI_API_KEY")

# In Colab, Secrets are not automatically exposed through os.environ.
# Fall back to Colab's userdata API when available.
if not API_KEY:
    try:
        from google.colab import userdata
        API_KEY = userdata.get("GEMINI_API_KEY")
    except Exception:
        API_KEY = None

if not API_KEY:
    raise RuntimeError(
        "GEMINI_API_KEY was not found. In Colab, add it under Secrets and "
        "enable notebook access for the secret."
    )

client = OpenAI(
    api_key=API_KEY,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)

# Stable, high-volume default. Override with GEMINI_MODEL if your project
# has access to a different Gemini Flash model.
MODEL = os.environ.get("GEMINI_MODEL", "gemini-3.5-flash-lite")

print("Client configured for:", MODEL)

## 2. Tool schemas

Each tool schema uses:
- a required-field list,
- explicit JSON types,
- enums where a small closed set is appropriate, and
- `additionalProperties: false` to prevent the model from inventing extra arguments.

The three tools are:

1. `lookup_course_reference` — returns a small local reference entry about function calling.
2. `convert_length` — converts a numeric distance among a fixed set of units.
3. `run_guarded_python` — evaluates one arithmetic expression after validation by an allowlist guardrail.

In [3]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "lookup_course_reference",
            "description": (
                "Look up a concise local reference for Week 4 concepts. "
                "Use only one of the supported topics and one detail level."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "topic": {
                        "type": "string",
                        "enum": ["tool_calling", "schema_design", "guardrails"],
                    },
                    "detail": {
                        "type": "string",
                        "enum": ["summary", "checklist"],
                    },
                },
                "required": ["topic", "detail"],
                "additionalProperties": False,
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "convert_length",
            "description": (
                "Convert one numeric length between supported units. "
                "Use this instead of estimating the conversion."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "value": {"type": "number"},
                    "from_unit": {
                        "type": "string",
                        "enum": ["millimeters", "centimeters", "meters", "kilometers", "inches", "feet", "miles"],
                    },
                    "to_unit": {
                        "type": "string",
                        "enum": ["millimeters", "centimeters", "meters", "kilometers", "inches", "feet", "miles"],
                    },
                },
                "required": ["value", "from_unit", "to_unit"],
                "additionalProperties": False,
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "run_guarded_python",
            "description": (
                "Evaluate exactly one numeric arithmetic expression. "
                "Only arithmetic is permitted; do not use names, imports, files, network access, "
                "function calls, attribute access, indexing, or process operations."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "task": {
                        "type": "string",
                        "enum": ["arithmetic"],
                    },
                    "code": {
                        "type": "string",
                        "description": "One Python arithmetic expression such as (17 * 23) + 11.",
                    },
                },
                "required": ["task", "code"],
                "additionalProperties": False,
            },
        },
    },
]

print(json.dumps(TOOLS, indent=2))

[
  {
    "type": "function",
    "function": {
      "name": "lookup_course_reference",
      "description": "Look up a concise local reference for Week 4 concepts. Use only one of the supported topics and one detail level.",
      "parameters": {
        "type": "object",
        "properties": {
          "topic": {
            "type": "string",
            "enum": [
              "tool_calling",
              "schema_design",
              "guardrails"
            ]
          },
          "detail": {
            "type": "string",
            "enum": [
              "summary",
              "checklist"
            ]
          }
        },
        "required": [
          "topic",
          "detail"
        ],
        "additionalProperties": false
      }
    }
  },
  {
    "type": "function",
    "function": {
      "name": "convert_length",
      "description": "Convert one numeric length between supported units. Use this instead of estimating the conversion.",
      "parameters": {


## 3. Tool implementations

### Local reference and conversion tools

These tools are intentionally deterministic and local. They do not need network access, which keeps the experiment focused on function calling rather than external API behavior.

In [4]:
COURSE_REFERENCE = {
    "tool_calling": {
        "summary": "A model becomes agent-like when it can choose a tool, receive its result, and decide what to do next.",
        "checklist": [
            "Send tool schemas with the model request.",
            "Inspect returned tool calls.",
            "Execute the selected tool.",
            "Append the tool result to the conversation.",
            "Call the model again so it can continue.",
        ],
    },
    "schema_design": {
        "summary": "Tight schemas constrain what arguments a model may send and reduce ambiguous or invented structures.",
        "checklist": [
            "Use explicit JSON types.",
            "Mark required fields.",
            "Use enums for closed choices.",
            "Reject unexpected properties.",
            "Keep structures as simple as the task allows.",
        ],
    },
    "guardrails": {
        "summary": "A guardrail checks model-suggested actions before execution and blocks categories outside the intended capability.",
        "checklist": [
            "Define an allowlist.",
            "Validate before execution.",
            "Return structured errors.",
            "Limit execution time.",
            "State clearly what is blocked.",
        ],
    },
}

METERS_PER_UNIT = {
    "millimeters": 0.001,
    "centimeters": 0.01,
    "meters": 1.0,
    "kilometers": 1000.0,
    "inches": 0.0254,
    "feet": 0.3048,
    "miles": 1609.344,
}

def lookup_course_reference(topic: str, detail: str) -> dict:
    return {
        "status": "ok",
        "topic": topic,
        "detail": detail,
        "content": COURSE_REFERENCE[topic][detail],
    }

def convert_length(value: float, from_unit: str, to_unit: str) -> dict:
    meters = value * METERS_PER_UNIT[from_unit]
    converted = meters / METERS_PER_UNIT[to_unit]
    return {
        "status": "ok",
        "input": {"value": value, "unit": from_unit},
        "output": {"value": converted, "unit": to_unit},
    }

### Guarded code runner

The code runner is deliberately narrow.

**Permitted**
- numeric constants,
- parentheses,
- arithmetic operators: `+`, `-`, `*`, `/`, `//`, `%`, and `**`,
- unary `+` and `-`.

**Blocked categories**
- filesystem access,
- network access,
- imports,
- function or method calls,
- variable/name access,
- attribute access,
- indexing/subscripts,
- loops, assignments, comprehensions, and control flow,
- model-supplied process execution.

The runner parses the expression with Python's AST module and rejects any node that is not on the allowlist. Execution occurs with empty built-ins in a separate worker process. The parent terminates the worker if it exceeds the time limit. This is a focused classroom guardrail, **not** a production sandbox.

In [5]:
ALLOWED_AST_NODES = (
    ast.Expression,
    ast.BinOp,
    ast.UnaryOp,
    ast.Constant,
    ast.Add,
    ast.Sub,
    ast.Mult,
    ast.Div,
    ast.FloorDiv,
    ast.Mod,
    ast.Pow,
    ast.USub,
    ast.UAdd,
)

def validate_arithmetic_expression(source: str) -> ast.AST:
    try:
        tree = ast.parse(source, mode="eval")
    except SyntaxError as exc:
        raise ValueError(f"Syntax error: {exc.msg}") from exc

    for node in ast.walk(tree):
        if not isinstance(node, ALLOWED_AST_NODES):
            raise ValueError(
                f"Blocked AST node: {type(node).__name__}. "
                "Only numeric arithmetic expressions are allowed."
            )
        if isinstance(node, ast.Constant) and not isinstance(node.value, (int, float)):
            raise ValueError("Only integer and floating-point numeric constants are allowed.")

    return tree

def _arithmetic_worker(source: str, output_queue: mp.Queue) -> None:
    try:
        tree = validate_arithmetic_expression(source)
        compiled = compile(tree, "<guarded-arithmetic>", "eval")
        result = eval(compiled, {"__builtins__": {}}, {})
        if not isinstance(result, (int, float)) or isinstance(result, bool):
            raise ValueError("Result must be numeric.")
        if isinstance(result, float) and not math.isfinite(result):
            raise ValueError("Result must be finite.")
        output_queue.put({"status": "ok", "result": result})
    except Exception as exc:
        output_queue.put({
            "status": "error",
            "error_type": type(exc).__name__,
            "message": str(exc),
        })

def run_guarded_python(task: str, code: str, timeout_seconds: float = 2.0) -> dict:
    if task != "arithmetic":
        return {
            "status": "error",
            "error_type": "UnsupportedTask",
            "message": "Only the arithmetic task is permitted.",
        }

    # Validate in the parent before any worker starts.
    try:
        validate_arithmetic_expression(code)
    except Exception as exc:
        return {
            "status": "error",
            "error_type": type(exc).__name__,
            "message": str(exc),
        }

    queue = mp.Queue()
    process = mp.Process(target=_arithmetic_worker, args=(code, queue))
    process.start()
    process.join(timeout_seconds)

    if process.is_alive():
        process.terminate()
        process.join()
        return {
            "status": "error",
            "error_type": "TimeoutError",
            "message": f"Execution exceeded {timeout_seconds} seconds.",
        }

    if queue.empty():
        return {
            "status": "error",
            "error_type": "ExecutionError",
            "message": "The worker ended without returning a result.",
        }

    return queue.get()

# Guardrail spot checks
guardrail_checks = {
    "allowed": run_guarded_python("arithmetic", "(17 * 23) + 11"),
    "blocked_function_call": run_guarded_python("arithmetic", "__import__('os').system('echo unsafe')"),
    "blocked_name": run_guarded_python("arithmetic", "open('/tmp/example.txt')"),
}
print(json.dumps(guardrail_checks, indent=2))

{
  "allowed": {
    "status": "ok",
    "result": 402
  },
  "blocked_function_call": {
    "status": "error",
    "error_type": "ValueError",
    "message": "Blocked AST node: Call. Only numeric arithmetic expressions are allowed."
  },
  "blocked_name": {
    "status": "error",
    "error_type": "ValueError",
    "message": "Blocked AST node: Call. Only numeric arithmetic expressions are allowed."
  }
}


## 4. Dispatcher and full agent loop

The loop below is the key agent behavior:

1. Send the conversation and tool schemas to the model.
2. If the model returns one or more tool calls, record them.
3. Execute each tool locally.
4. Append each structured tool result to the conversation.
5. Send the updated conversation back to the model.
6. Repeat until the model returns a normal final answer or the maximum number of rounds is reached.

Tool failures are returned to the model as data. They do not crash the agent loop.

In [ ]:
def dispatch_tool(name: str, arguments: dict) -> dict:
    try:
        if name == "lookup_course_reference":
            return lookup_course_reference(**arguments)
        if name == "convert_length":
            return convert_length(**arguments)
        if name == "run_guarded_python":
            return run_guarded_python(**arguments)
        return {
            "status": "error",
            "error_type": "UnknownTool",
            "message": f"Unknown tool: {name}",
        }
    except Exception as exc:
        return {
            "status": "error",
            "error_type": type(exc).__name__,
            "message": str(exc),
        }

def model_request(messages, max_retries=5):
    """Call Gemini with bounded exponential backoff for transient/rate-limit failures."""
    delay = 2

    for attempt in range(max_retries):
        try:
            return client.chat.completions.create(
                model=MODEL,
                messages=messages,
                tools=TOOLS,
                tool_choice="auto",
                temperature=0,
            )
        except Exception as exc:
            message = str(exc)
            transient = (
                "429" in message
                or "RESOURCE_EXHAUSTED" in message
                or "rate" in message.lower()
                or "500" in message
                or "502" in message
                or "503" in message
                or "504" in message
            )

            if not transient or attempt == max_retries - 1:
                raise

            print(
                f"Transient API failure ({type(exc).__name__}). "
                f"Retrying in {delay} seconds..."
            )
            time.sleep(delay)
            delay = min(delay * 2, 30)

RESPONSE_CACHE: dict[str, dict] = {}

def run_agent(user_query: str, use_cache: bool = True, max_rounds: int = 8) -> dict:
    if use_cache and user_query in RESPONSE_CACHE:
        print("Using cached result for this query.")
        return RESPONSE_CACHE[user_query]

    messages = [
        {
            "role": "system",
            "content": (
                "You are a careful Week 4 tool-calling assistant. "
                "Use the supplied tools whenever the request depends on them. "
                "If a tool returns a structured error, inspect it and decide whether a safe corrected retry is appropriate. "
                "Never claim a tool succeeded when its result reports an error."
            ),
        },
        {"role": "user", "content": user_query},
    ]

    tool_log = []

    for round_number in range(1, max_rounds + 1):
        response = model_request(messages)
        assistant_message = response.choices[0].message

        # Preserve the assistant tool-call message in the conversation.
        messages.append(assistant_message.model_dump(exclude_none=True))

        if not assistant_message.tool_calls:
            result = {
                "query": user_query,
                "answer": assistant_message.content,
                "tool_log": tool_log,
                "rounds": round_number,
            }
            RESPONSE_CACHE[user_query] = result
            return result

        for tool_call in assistant_message.tool_calls:
            name = tool_call.function.name
            try:
                arguments = json.loads(tool_call.function.arguments)
            except json.JSONDecodeError as exc:
                arguments = {}
                tool_result = {
                    "status": "error",
                    "error_type": "JSONDecodeError",
                    "message": str(exc),
                }
            else:
                tool_result = dispatch_tool(name, arguments)

            succeeded = tool_result.get("status") == "ok"
            tool_log.append({
                "round": round_number,
                "tool": name,
                "arguments": arguments,
                "succeeded": succeeded,
                "result": tool_result,
            })

            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": json.dumps(tool_result),
            })

    raise RuntimeError(f"Agent exceeded max_rounds={max_rounds} without a final answer.")

def print_run(result: dict) -> None:
    print("QUERY:")
    print(result["query"])
    print("\nTOOL CALL LOG:")
    if not result["tool_log"]:
        print("  (no tool calls)")
    for i, entry in enumerate(result["tool_log"], start=1):
        print(f"  {i}. tool={entry['tool']}")
        print(f"     arguments={json.dumps(entry['arguments'])}")
        print(f"     succeeded={entry['succeeded']}")
        print(f"     result={json.dumps(entry['result'])}")
    print("\nFINAL ANSWER:")
    print(result["answer"])

## 5. Evaluation

The following queries are designed to exercise every tool. The fourth query requires two tools in sequence.

Because these are live model calls, the saved notebook output after execution is the measured evidence for which tool Gemini selected and what arguments it generated.

In [ ]:
evaluation_queries = [
    "Use the course reference tool to give me the checklist for schema design.",
    "Convert 5280 feet to miles using the conversion tool.",
    "Use the guarded Python tool to calculate (17 * 23) + 11.",
    (
        "First use the guarded Python tool to calculate 144 * 3. "
        "Then convert that result from centimeters to meters using the conversion tool. "
        "Use both tools rather than doing either step mentally."
    ),
]

evaluation_results = []

for index, query in enumerate(evaluation_queries):
    print("=" * 90)
    result = run_agent(query)
    evaluation_results.append(result)
    print_run(result)
    print()

    # Space out independent evaluation queries to reduce burst rate-limit pressure.
    # Skip the sleep after the final query.
    if index < len(evaluation_queries) - 1:
        time.sleep(5)

### Evaluation checklist

After running the previous cell, verify from the printed log that:

- Query 1 called `lookup_course_reference`.
- Query 2 called `convert_length`.
- Query 3 called `run_guarded_python`.
- Query 4 called both `run_guarded_python` and `convert_length` in sequence.
- Every logged call shows the arguments and a `succeeded` value.

If the model chooses a different but valid path on a live run, report what actually happened rather than editing the output to match expectations.

In [8]:
summary_rows = []
for result in evaluation_results:
    summary_rows.append({
        "query": result["query"],
        "tools": [entry["tool"] for entry in result["tool_log"]],
        "all_calls_succeeded": all(entry["succeeded"] for entry in result["tool_log"]),
        "rounds": result["rounds"],
    })

for row in summary_rows:
    print(json.dumps(row, indent=2))

{
  "query": "Use the course reference tool to give me the checklist for schema design.",
  "tools": [
    "lookup_course_reference"
  ],
  "all_calls_succeeded": true,
  "rounds": 2
}
{
  "query": "Convert 5280 feet to miles using the conversion tool.",
  "tools": [
    "convert_length"
  ],
  "all_calls_succeeded": true,
  "rounds": 2
}
{
  "query": "Use the guarded Python tool to calculate (17 * 23) + 11.",
  "tools": [
    "run_guarded_python"
  ],
  "all_calls_succeeded": true,
  "rounds": 2
}
{
  "query": "First use the guarded Python tool to calculate 144 * 3. Then convert that result from centimeters to meters using the conversion tool. Use both tools rather than doing either step mentally.",
  "tools": [
    "run_guarded_python",
    "convert_length"
  ],
  "all_calls_succeeded": true,
  "rounds": 3
}


## 6. Part 4 — Real failure and recovery

The failure case is a **runtime error in a tool**, which is explicitly allowed by the assignment.

The schema-valid call below requests `10 / 0`. Division is an allowed arithmetic operation, so the AST guardrail correctly permits the expression to reach execution. Python then raises `ZeroDivisionError`.

The tool catches that exception and converts it into a structured error. The agent loop sends that error back to the model instead of crashing. The user prompt explicitly permits a corrected retry using `10 / 2`, allowing the model to demonstrate recovery.

This is documented in GitHub issue **#15**.

In [9]:
failure_query = (
    "Use the guarded Python tool to calculate 10 / 0. "
    "That first call is intentionally expected to fail. "
    "After you receive the structured runtime error, recover by retrying with 10 / 2, "
    "then explain briefly what failed and report the successful corrected result."
)

failure_result = run_agent(failure_query, use_cache=False)
print_run(failure_result)

QUERY:
Use the guarded Python tool to calculate 10 / 0. That first call is intentionally expected to fail. After you receive the structured runtime error, recover by retrying with 10 / 2, then explain briefly what failed and report the successful corrected result.

TOOL CALL LOG:
  1. tool=run_guarded_python
     arguments={"task": "arithmetic", "code": "10 / 0"}
     succeeded=False
     result={"status": "error", "error_type": "ZeroDivisionError", "message": "division by zero"}
  2. tool=run_guarded_python
     arguments={"code": "10 / 2", "task": "arithmetic"}
     succeeded=True
     result={"status": "ok", "result": 5.0}

FINAL ANSWER:
The initial attempt to calculate `10 / 0` failed because it triggered a `ZeroDivisionError`, as division by zero is mathematically undefined and not permitted by the tool. After receiving this error, I retried the operation using `10 / 2`, which successfully returned a result of `5.0`.


### Failure analysis

**Schema used:** `run_guarded_python` requires a string `task` whose enum is `arithmetic` and a string `code`. Extra properties are forbidden.

**Bad call:** `{"task": "arithmetic", "code": "10 / 0"}`

**Why it failed:** The arguments satisfy the JSON schema and the code satisfies the AST allowlist, but division by zero is invalid at runtime. The guarded worker returns a structured `ZeroDivisionError`.

**Recovery:** The model receives that error through the normal tool-result message and retries with `{"task": "arithmetic", "code": "10 / 2"}`.

**Fix category:** Retry after a structured runtime error. No schema correction is needed because the original arguments are schema-valid.

The live log above is the evidence. It should show both the failed and successful calls. If it does not, rerun only this failure experiment and document the actual behavior rather than claiming a recovery that did not occur.

## 7. Conclusions

This experiment demonstrates the point at which ordinary model completion becomes a simple agent loop: the model can choose a constrained action, observe the result, and make another decision.

The schemas limit the model's action space, while the guarded code runner adds a separate safety boundary before model-suggested code is executed. The runtime-error experiment also shows why structured tool errors matter: a tool can fail without crashing the whole interaction, and the model can use the returned error as new context for recovery.

### Safety limitation

The arithmetic runner is only a focused instructional control. It is not a complete sandbox and should not be used as a production code-execution service.

## 8. AI assistance disclosure

ChatGPT was used to scaffold the notebook structure, tool schemas, agent loop, guardrail design, evaluation plan, README documentation, and the GitHub failure/recovery issue. The live Gemini calls, tool selections, arguments, outputs, and observations are to be produced by running this notebook. Final analysis of those measured results remains the student's responsibility.